# 02 — Bronze Ingestion

**Project:** Orchestrated Incremental Orders Pipeline with Databricks Workflows  
**Layer:** Bronze  
**Source:** `orders_workflow_project.source_orders_batch_<batch_id>`  
**Target:** `orders_workflow_project.bronze_orders_raw`

This notebook ingests one source batch at a time into the Bronze Delta table.

The notebook is designed to be executed by a Databricks Workflow / Job using parameters:

- `pipeline_run_id`
- `batch_id`

The Bronze layer stores all incoming order events using append logic and keeps the full historical event log.

In [0]:
dbutils.widgets.text("pipeline_run_id", "manual_run", "Pipeline Run ID")

dbutils.widgets.dropdown(
    "batch_id",
    "1",
    ["1", "2", "3"],
    "Batch ID"
)

pipeline_run_id = dbutils.widgets.get("pipeline_run_id")
batch_id = int(dbutils.widgets.get("batch_id"))

print(f"Pipeline Run ID: {pipeline_run_id}")
print(f"Selected Batch ID: {batch_id}")

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

In [0]:
spark.sql("USE SCHEMA orders_workflow_project")

In [0]:
source_table = f"orders_workflow_project.source_orders_batch_{batch_id}"
bronze_table = "orders_workflow_project.bronze_orders_raw"
audit_table = "orders_workflow_project.pipeline_run_audit"

print(f"Source table: {source_table}")
print(f"Bronze table: {bronze_table}")
print(f"Audit table: {audit_table}")

In [0]:
source_df = spark.table(source_table)

display(
    source_df.orderBy("order_id", "event_ts")
)

In [0]:
bronze_df = (
    source_df
    .withColumn("bronze_ingestion_ts", current_timestamp())
    .withColumn("source_batch_table", lit(source_table))
)

display(bronze_df)

In [0]:
bronze_exists = spark.catalog.tableExists(bronze_table)

print(f"Bronze table exists: {bronze_exists}")

In [0]:
if bronze_exists:
    existing_batch_count = (
        spark.table(bronze_table)
        .filter(col("batch_id") == batch_id)
        .count()
    )
else:
    existing_batch_count = 0

print(f"Existing records for batch_id {batch_id}: {existing_batch_count}")

In [0]:
if existing_batch_count == 0:
    bronze_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(bronze_table)

    records_processed = bronze_df.count()
    task_status = "SUCCESS"
    task_message = f"Batch {batch_id} ingested successfully into Bronze."

else:
    records_processed = 0
    task_status = "SKIPPED"
    task_message = f"Batch {batch_id} was already ingested. Skipping append."

print(task_status)
print(task_message)
print(f"Records processed: {records_processed}")

In [0]:
pipeline_run_id_sql = pipeline_run_id.replace("'", "''")
task_name_sql = f"02_bronze_ingestion_batch_{batch_id}"
status_sql = task_status.replace("'", "''")
message_sql = task_message.replace("'", "''")

spark.sql(f"""
INSERT INTO {audit_table}
SELECT
    '{pipeline_run_id_sql}' AS pipeline_run_id,
    '{task_name_sql}' AS task_name,
    '{status_sql}' AS status,
    CAST({int(records_processed)} AS BIGINT) AS records_processed,
    '{message_sql}' AS message,
    current_timestamp() AS processed_ts
""")

print("Audit record inserted successfully.")

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        customer_id,
        order_status,
        order_amount,
        batch_id,
        event_ts,
        bronze_ingestion_ts,
        source_batch_table
    FROM orders_workflow_project.bronze_orders_raw
    ORDER BY batch_id, order_id, event_ts
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        batch_id,
        COUNT(*) AS total_records
    FROM orders_workflow_project.bronze_orders_raw
    GROUP BY batch_id
    ORDER BY batch_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT
        order_id,
        COUNT(*) AS total_events,
        COLLECT_LIST(order_status) AS statuses
    FROM orders_workflow_project.bronze_orders_raw
    GROUP BY order_id
    ORDER BY order_id
    """)
)

In [0]:
display(
    spark.sql("""
    SELECT *
    FROM orders_workflow_project.pipeline_run_audit
    ORDER BY processed_ts DESC
    """)
)